# Day 54 — Positional encodings: sinusoidal, RoPE, ALiBi, and the FlexAttention idea

Attention has no idea what order its inputs are in: shuffle the tokens and the output
shuffles with them, to the last bit. Every transformer puts the order back by hand, and
the three common ways differ most in what happens past the length they were trained on.

In [1]:
import math

import numpy as np

# ================================================================ 1. attention again
def softmax(x, axis=-1):
    x = np.asarray(x, dtype=float)
    m = np.max(x, axis=axis, keepdims=True)
    e = np.exp(x - m)
    return e / e.sum(axis=axis, keepdims=True)


def attention(Q, K, V, mask=None, bias=None):
    """softmax(QK^T/sqrt(d) + bias) V, with bias the hook ALiBi needs."""
    d = Q.shape[-1]
    s = Q @ K.T / math.sqrt(d)
    if bias is not None:
        s = s + bias
    if mask is not None:
        s = np.where(mask, s, -np.inf)
    w = softmax(s, axis=-1)
    return w @ V, w


def permutation_test(n=8, d=16, seed=0):
    """Attention is permutation-equivariant: permute the input, permute the output.

    Nothing in softmax(QK^T)V refers to an index, so the model literally cannot tell
    "the cat sat" from "sat cat the" until something puts the order back.
    """
    rng = np.random.default_rng(seed)
    X = rng.standard_normal((n, d))
    perm = rng.permutation(n)
    out, _ = attention(X, X, X)
    out_perm, _ = attention(X[perm], X[perm], X[perm])
    return float(np.abs(out[perm] - out_perm).max())


print('permute then attend vs attend then permute:', f'{permutation_test():.1e}')
print('-> attention is a set function until something puts the order back')

permute then attend vs attend then permute: 2.2e-16
-> attention is a set function until something puts the order back


## Sinusoidal: a table of sines

Shifting by `k` is a fixed rotation of each (sin, cos) pair — the same matrix at every
position, which is what makes "five tokens back" a single linear relationship.

In [2]:
# ================================================================ 2. sinusoidal
def sinusoidal(n, d, base=10000.0):
    """The 2017 table: pos / base^(2i/d) through sin and cos, one row per position."""
    pos = np.arange(n)[:, None]
    i = np.arange(0, d, 2)[None, :]
    ang = pos / (base ** (i / d))
    pe = np.zeros((n, d))
    pe[:, 0::2] = np.sin(ang)
    pe[:, 1::2] = np.cos(ang)
    return pe


def sinusoidal_shift_is_linear(n=64, d=32, k=5):
    """Shifting by k is a fixed rotation of each (sin, cos) pair - the same matrix for
    every position, which is what makes relative offsets learnable at all."""
    pe = sinusoidal(n, d)
    i = np.arange(0, d, 2)
    theta = k / (10000.0 ** (i / d))
    rot = np.zeros((n - k, d))
    s, c = np.sin(theta), np.cos(theta)
    rot[:, 0::2] = pe[:-k, 0::2] * c + pe[:-k, 1::2] * s
    rot[:, 1::2] = pe[:-k, 1::2] * c - pe[:-k, 0::2] * s
    return float(np.abs(rot - pe[k:]).max())


def sinusoidal_similarity(n=64, d=64):
    """Dot product between the encoding of position 0 and every other position."""
    pe = sinusoidal(n, d)
    return (pe @ pe[0]) / d


print(np.round(sinusoidal(8, 8), 4))
print()
print('shift-by-k is one fixed rotation, error:', f'{sinusoidal_shift_is_linear():.1e}')
sim = sinusoidal_similarity()
print('similarity to position 0 at distance 1, 2, 4, 8, 16, 32:',
      ' '.join(f'{sim[t]:+.3f}' for t in (1, 2, 4, 8, 16, 32)))

[[ 0.      1.      0.      1.      0.      1.      0.      1.    ]
 [ 0.8415  0.5403  0.0998  0.995   0.01    1.      0.001   1.    ]
 [ 0.9093 -0.4161  0.1987  0.9801  0.02    0.9998  0.002   1.    ]
 [ 0.1411 -0.99    0.2955  0.9553  0.03    0.9996  0.003   1.    ]
 [-0.7568 -0.6536  0.3894  0.9211  0.04    0.9992  0.004   1.    ]
 [-0.9589  0.2837  0.4794  0.8776  0.05    0.9988  0.005   1.    ]
 [-0.2794  0.9602  0.5646  0.8253  0.06    0.9982  0.006   1.    ]
 [ 0.657   0.7539  0.6442  0.7648  0.0699  0.9976  0.007   1.    ]]

shift-by-k is one fixed rotation, error: 3.3e-15
similarity to position 0 at distance 1, 2, 4, 8, 16, 32: +0.483 +0.442 +0.374 +0.350 +0.303 +0.306


## RoPE: the position is a rotation

Rotate each (even, odd) channel pair of `q` and `k` by an angle proportional to the
position, and the dot product depends only on the distance. A rotation does not change
lengths, and `V` is never touched.

In [3]:
# ================================================================ 3. RoPE
def rope_angles(d, pos, base=10000.0):
    i = np.arange(0, d, 2)
    return np.outer(np.asarray(pos, dtype=float), 1.0 / (base ** (i / d)))


def rope(x, pos, base=10000.0):
    """Rotate each (even, odd) pair of channels by an angle proportional to position.

    No vector is added to anything: the position is applied to q and k as a rotation,
    so it changes the direction of the query and the key, not their length.
    """
    x = np.atleast_2d(np.asarray(x, dtype=float))
    ang = rope_angles(x.shape[-1], pos, base)
    c, s = np.cos(ang), np.sin(ang)
    out = np.empty_like(x)
    out[..., 0::2] = x[..., 0::2] * c - x[..., 1::2] * s
    out[..., 1::2] = x[..., 1::2] * c + x[..., 0::2] * s
    return out


def rope_is_relative(d=64, m=3, n=11, shift=97, seed=0):
    """<RoPE(q, m), RoPE(k, n)> depends only on n - m, not on m and n themselves."""
    rng = np.random.default_rng(seed)
    q, k = rng.standard_normal(d), rng.standard_normal(d)
    a = float(rope(q, [m])[0] @ rope(k, [n])[0])
    b = float(rope(q, [m + shift])[0] @ rope(k, [n + shift])[0])
    return a, b, abs(a - b)


def rope_wavelengths(d=128, base=10000.0):
    """How many tokens each channel pair takes to complete one full turn."""
    i = np.arange(0, d, 2)
    freq = 1.0 / (base ** (i / d))
    return 2 * math.pi / freq


def rope_context_scaling(d=128, base=10000.0, train=4096, target=32768):
    """Two ways to make a model trained to `train` work at `target`.

    Position interpolation squeezes the positions; raising the base stretches the
    wavelengths.  Both aim at the same thing: no channel should see an angle it never
    saw in training.
    """
    wl = rope_wavelengths(d, base)
    slowest = wl.max()
    need = base * (target / train) ** (d / (d - 2))
    return {'slowest_wavelength': float(slowest),
            'turns_at_train': train / slowest,
            'turns_at_target': target / slowest,
            'interp_factor': train / target,
            'ntk_base': float(need)}


a, b, gap = rope_is_relative()
print(f'<RoPE(q,3), RoPE(k,11)>   = {a:+.6f}')
print(f'<RoPE(q,100), RoPE(k,108)> = {b:+.6f}   difference {gap:.1e}')
wl = rope_wavelengths()
print(f'wavelengths at d=128: fastest {wl.min():.2f} tokens, slowest {wl.max():,.0f}')
sc = rope_context_scaling()
print(f"trained at 4,096 the slowest pair turns {sc['turns_at_train']:.2f} of a circle; "
      f"32,768 would need {sc['turns_at_target']:.2f}")
print(f"position interpolation factor {sc['interp_factor']:.3f}, or raise the base to "
      f"{sc['ntk_base']:,.0f}")

<RoPE(q,3), RoPE(k,11)>   = -5.300637
<RoPE(q,100), RoPE(k,108)> = -5.300637   difference 2.7e-15
wavelengths at d=128: fastest 6.28 tokens, slowest 54,410
trained at 4,096 the slowest pair turns 0.08 of a circle; 32,768 would need 0.60
position interpolation factor 0.125, or raise the base to 82,685


## ALiBi: a penalty on distance

One slope per head, added straight to the scores. Nothing to embed, nothing to train,
and each head becomes a soft window of its own width.

In [4]:
# ================================================================ 4. ALiBi
def alibi_slopes(h):
    """The published geometric sequence: 2^(-8/h), 2^(-16/h), ... for h heads."""
    start = 2.0 ** (-8.0 / h)
    return np.array([start ** (i + 1) for i in range(h)])


def alibi_bias(n, slope):
    """-slope * (distance), added to the scores before the softmax."""
    pos = np.arange(n)
    dist = pos[None, :] - pos[:, None]          # key - query, negative into the past
    return slope * dist


def alibi_effective_window(n=512, h=8, tail=0.01):
    """How far back each head still puts `tail` of its weight, with flat scores.

    With no content signal the weights are exp(-slope * distance) normalised, which is
    a geometric decay: each head is a soft window of its own width.
    """
    rows = []
    for sl in alibi_slopes(h):
        b = alibi_bias(n, sl)[-1]
        w = softmax(np.where(np.arange(n) <= n - 1, b, -np.inf))
        cum = np.cumsum(w[::-1])                # from the newest token backwards
        k = int(np.searchsorted(cum, 1 - tail) + 1)
        rows.append({'slope': float(sl), 'window': k})
    return rows


for i, r in enumerate(alibi_effective_window()):
    print(f"head {i}  slope {r['slope']:.4f}  99% of the weight within {r['window']:,} tokens")
print('(the last row is clipped by the 512-token probe, not by the slope)')

head 0  slope 0.5000  99% of the weight within 10 tokens
head 1  slope 0.2500  99% of the weight within 19 tokens
head 2  slope 0.1250  99% of the weight within 37 tokens
head 3  slope 0.0625  99% of the weight within 74 tokens
head 4  slope 0.0312  99% of the weight within 148 tokens
head 5  slope 0.0156  99% of the weight within 293 tokens
head 6  slope 0.0078  99% of the weight within 458 tokens
head 7  slope 0.0039  99% of the weight within 497 tokens
(the last row is clipped by the 512-token probe, not by the slope)


## What survives past the trained length

All three are formulas, so all three can be evaluated at an unseen position — unlike a
learned table. Being evaluable is not the same as being meaningful.

In [5]:
# ================================================================ 6. seeded tables
def far_token_wins(train=32, far=512, d=64, trials=200, seed=0):
    """How often a token beyond the trained length out-scores everything inside it.

    No model is trained here; this is a property of the encoding itself.  For each
    random (q, k) pair, compare the best score at a distance the model has seen with
    the best score at a distance it has not.
    """
    rng = np.random.default_rng(seed)
    dist = np.arange(1, far + 1)
    near, far_m = dist <= train, dist > train
    pe = sinusoidal(far + 1, d)
    slope = alibi_slopes(8)[3]
    out = {'sinusoidal': 0, 'rope': 0, 'alibi': 0}
    for _ in range(trials):
        q, k = rng.standard_normal(d), rng.standard_normal(d)
        s_sin = np.array([(q + pe[t]) @ (k + pe[0]) for t in dist]) / math.sqrt(d)
        s_rope = np.array([rope(q, [t])[0] @ rope(k, [0])[0] for t in dist]) / math.sqrt(d)
        s_ali = (q @ k) / math.sqrt(d) - slope * dist
        for name, sc in (('sinusoidal', s_sin), ('rope', s_rope), ('alibi', s_ali)):
            if sc[far_m].max() > sc[near].max():
                out[name] += 1
    return {k2: v / trials for k2, v in out.items()}


fw = far_token_wins()
for k, v in fw.items():
    print(f'{k:>12}: a token past the trained length wins {v * 100:5.1f}% of the time')

  sinusoidal: a token past the trained length wins  10.0% of the time
        rope: a token past the trained length wins  93.5% of the time
       alibi: a token past the trained length wins   0.0% of the time


## FlexAttention: the scheme is an argument

Every encoding above touches one of two things: the value of a score, or whether a pair
is allowed. Hand attention those two as functions and causal masking, ALiBi, sliding
windows and soft-capping stop being separate kernels — and whole tiles with nothing
allowed inside them never get computed.

In [6]:
# ================================================================ 5. FlexAttention
def flex_attention(Q, K, V, score_mod=None, mask_mod=None):
    """The FlexAttention idea in nine lines of numpy.

    One function edits the score, one function says which pairs are allowed, and every
    positional scheme in this file becomes an argument instead of a kernel.
    """
    n, d = Q.shape
    m = K.shape[0]
    s = Q @ K.T / math.sqrt(d)
    qi = np.arange(n)[:, None]
    kj = np.arange(m)[None, :]
    if score_mod is not None:
        s = score_mod(s, qi, kj)
    if mask_mod is not None:
        s = np.where(mask_mod(qi, kj), s, -np.inf)
    w = softmax(s, axis=-1)
    return w @ V, w


def mod_causal():
    return lambda qi, kj: kj <= qi


def mod_sliding(window):
    return lambda qi, kj: (kj <= qi) & (qi - kj < window)


def mod_alibi(slope):
    return lambda s, qi, kj: s + slope * (kj - qi)


def mod_softcap(cap):
    return lambda s, qi, kj: cap * np.tanh(s / cap)


def block_sparsity(n=1024, block=128, window=256):
    """What a block mask buys: whole tiles that are entirely masked are never computed.

    FlexAttention's BlockMask is this table - the fraction of (query block, key block)
    tiles with nothing allowed inside them.
    """
    nb = n // block
    qi = np.arange(n)[:, None]
    kj = np.arange(n)[None, :]
    rows = []
    for name, allowed in (('full', np.ones((n, n), dtype=bool)),
                          ('causal', mod_causal()(qi, kj)),
                          (f'sliding {window}', mod_sliding(window)(qi, kj))):
        live = 0
        for a in range(nb):
            for b in range(nb):
                if allowed[a * block:(a + 1) * block, b * block:(b + 1) * block].any():
                    live += 1
        rows.append({'name': name, 'live': live, 'total': nb * nb,
                     'skipped': 1 - live / (nb * nb)})
    return rows


rng = np.random.default_rng(4)
Q, K, V = (rng.standard_normal((12, 16)), rng.standard_normal((12, 16)),
           rng.standard_normal((12, 8)))
msk = np.tril(np.ones((12, 12), dtype=bool))
ref, _ = attention(Q, K, V, mask=msk)
got, _ = flex_attention(Q, K, V, mask_mod=mod_causal())
sl = alibi_slopes(8)[0]
ref2, _ = attention(Q, K, V, mask=msk, bias=alibi_bias(12, sl))
got2, _ = flex_attention(Q, K, V, score_mod=mod_alibi(sl), mask_mod=mod_causal())
print('causal:         hand vs flex', np.abs(ref - got).max())
print('causal + ALiBi: hand vs flex', np.abs(ref2 - got2).max())
print()
for r in block_sparsity():
    print(f"{r['name']:>14}: {r['live']:>3} of {r['total']} tiles computed, "
          f"{r['skipped'] * 100:5.1f}% skipped")

causal:         hand vs flex 0.0
causal + ALiBi: hand vs flex 0.0

          full:  64 of 64 tiles computed,   0.0% skipped
        causal:  36 of 64 tiles computed,  43.8% skipped
   sliding 256:  21 of 64 tiles computed,  67.2% skipped


## The silent failure: RoPE at the wrong position while decoding

Appending one token to a cache of five puts the new query at position 5. Rotating it at
position 0 — because it is the first row of this step — is one line, and the result is
still a finite set of weights that sums to 1.

In [7]:
def decode_offset_bug(d=64, n=6, seed=1):
    """The cache bug: RoPE applied at the wrong position when a token is appended.

    During decoding the new token's absolute position is len(cache); using 0 instead
    (because the new query is "the first row of this step") is a one-line mistake that
    produces finite, plausible numbers.
    """
    rng = np.random.default_rng(seed)
    X = rng.standard_normal((n, d))
    pos = np.arange(n)
    Kr = rope(X, pos)
    q = rng.standard_normal(d)
    right = rope(q[None, :], [n - 1])[0] @ Kr.T / math.sqrt(d)
    wrong = rope(q[None, :], [0])[0] @ Kr.T / math.sqrt(d)
    return (softmax(right), softmax(wrong),
            float(np.abs(softmax(right) - softmax(wrong)).max()))


right, wrong, gap = decode_offset_bug()
print('correct position:', np.round(right, 4))
print('position 0:      ', np.round(wrong, 4))
print(f'largest difference: {gap:.4f}   (both rows sum to {wrong.sum():.1f})')

correct position: [0.1669 0.0351 0.1301 0.3161 0.1093 0.2425]
position 0:       [0.0808 0.0391 0.1214 0.3489 0.1837 0.2262]
largest difference: 0.0862   (both rows sum to 1.0)


## The assertions

In [8]:
# ================================================================ 7. assertions
def check():
    # attention cannot see order at all
    assert permutation_test() < 1e-12

    # sinusoidal: shifting by k is one fixed rotation, independent of position
    assert sinusoidal_shift_is_linear() < 1e-12

    # RoPE: the score depends only on the distance
    a, b, gap = rope_is_relative()
    assert gap < 1e-9 and abs(a) > 1e-6
    # ... and it is a rotation, so it does not change lengths
    rng = np.random.default_rng(3)
    v = rng.standard_normal(64)
    assert abs(np.linalg.norm(rope(v, [17])[0]) - np.linalg.norm(v)) < 1e-12

    # ALiBi: a bigger slope means a shorter window
    rows = alibi_effective_window()
    assert rows[0]['window'] < rows[-1]['window']   # steeper slope, shorter window

    # FlexAttention reproduces the hand-written versions exactly
    rng = np.random.default_rng(4)
    Q, K, V = (rng.standard_normal((12, 16)), rng.standard_normal((12, 16)),
               rng.standard_normal((12, 8)))
    ref, _ = attention(Q, K, V, mask=np.tril(np.ones((12, 12), dtype=bool)))
    got, _ = flex_attention(Q, K, V, mask_mod=mod_causal())
    assert np.abs(ref - got).max() == 0.0
    sl = alibi_slopes(8)[0]
    ref2, _ = attention(Q, K, V, mask=np.tril(np.ones((12, 12), dtype=bool)),
                        bias=alibi_bias(12, sl))
    got2, _ = flex_attention(Q, K, V, score_mod=mod_alibi(sl), mask_mod=mod_causal())
    assert np.abs(ref2 - got2).max() == 0.0

    # ALiBi can never let a far token win; the oscillating schemes can
    fw = far_token_wins(trials=40)
    assert fw['alibi'] == 0.0 and fw['rope'] > 0.0

    # a block mask skips strictly more as the window gets smaller
    bs = block_sparsity()
    assert bs[0]['skipped'] == 0 and bs[1]['skipped'] < bs[2]['skipped']

    # the decode offset bug is silent: finite, normalised, and wrong
    r, w, gap2 = decode_offset_bug()
    assert np.isfinite(w).all() and abs(w.sum() - 1) < 1e-12 and gap2 > 0.05
    return True


print('all assertions passed' if check() else 'FAILED')

all assertions passed
